In [28]:
import pandas as pd
import numpy as np

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_log_error as RMSE


data_path = "train.csv"
data = pd.read_csv(data_path)
target_thing = np.log1p(data["hours_spent"])

input_data = data.drop(columns=[
    "project_id",
    "hours_spent",
    "log_hours",
    "hours_bucket"
])


qualitative_features = input_data.select_dtypes(include=["object", "string", "category"]).columns.tolist() 
#print(qualitative_features) => ['commit_cadence', 'primary_language', 'framework', 'has_custom_art', 'hosting_type', 'is_mobile_responsive', 'project_category', 'is_multiplayer', 'mentions_ai_api', 'is_clone_or_tutorial', 'author_prior_projects', 'author_account_age_years']

for col in qualitative_features: #fill in missing stuff just in case
    input_data[col] = input_data[col].fillna("Unknown").astype(str)

input_training, input_validation, output_training, output_validation = train_test_split(input_data, target_thing, test_size=0.2, random_state=42)

magicModel = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    loss_function="RMSE",
    verbose=100,
    early_stopping_rounds=100
)

magicModel.fit(
    input_training,
    output_training,
    cat_features=qualitative_features,
    eval_set=(input_validation, output_validation)
)

predictionsG=np.expm1(magicModel.predict(input_validation))
predictionsG = np.maximum(predictionsG, 0) #no negatives

true_hours = np.expm1(output_validation)

scoreG=RMSE(true_hours,predictionsG)

print("the score:", scoreG)

results=[]

for category in input_data["project_category"].unique():

    training_check=input_training["project_category"] == category
    validation_check=input_validation["project_category"] == category

    input_category_training = input_training.loc[training_check]
    input_category_validation = input_validation.loc[validation_check]

    output_category_training = output_training.loc[training_check]
    output_category_validation = output_validation.loc[validation_check]
    print(input_category_training.columns[57])
    category_columns = input_category_training.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    magicModel = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=100,
        early_stopping_rounds=100
    )

    if len(input_category_training) < 50:
        print(f"Skipping {category}: insufficient training data")
        continue

    if len(input_category_validation) == 0:
        print(f"Skipping {category}: no validation data")
        continue

    if len(output_category_training) == 0:
        print(f"Skipping {category}: no training labels")
        continue

    magicModel.fit(
        input_category_training,
        output_category_training,
        cat_features=category_columns,
        eval_set=(input_category_validation, output_category_validation)
    )

    predictions=np.expm1(magicModel.predict(input_category_validation))
    predictions = np.maximum(predictions, 0) #no negatives

    true_hours = np.expm1(output_category_validation)

    score=RMSE(true_hours,predictions)

    qualpredictionG = predictionsG[validation_check.to_numpy()]
    qualscoreG = RMSE(true_hours, qualpredictionG)

    results.append({
        "category": category,
        "train_size": len(input_category_training),
        "test_size": len(input_category_validation),
        "normal_score": qualscoreG,
        "sorted_score": score,
        "improvement": qualscoreG-score
    })
results = pd.DataFrame(results).sort_values("improvement",ascending=False)
print(results.to_string(index=False))

0:	learn: 0.9654659	test: 0.8962037	best: 0.8962037 (0)	total: 9.76ms	remaining: 9.74s
100:	learn: 0.6636908	test: 0.6770706	best: 0.6770706 (100)	total: 547ms	remaining: 4.87s
200:	learn: 0.6196252	test: 0.6653937	best: 0.6653937 (200)	total: 1.1s	remaining: 4.37s
300:	learn: 0.5922211	test: 0.6628563	best: 0.6628563 (300)	total: 1.74s	remaining: 4.03s
400:	learn: 0.5645951	test: 0.6594709	best: 0.6592850 (399)	total: 2.23s	remaining: 3.33s
500:	learn: 0.5418152	test: 0.6566638	best: 0.6566638 (500)	total: 2.8s	remaining: 2.79s
600:	learn: 0.5233783	test: 0.6546460	best: 0.6546460 (600)	total: 3.27s	remaining: 2.17s
700:	learn: 0.5036165	test: 0.6533969	best: 0.6533791 (698)	total: 3.74s	remaining: 1.6s
800:	learn: 0.4876503	test: 0.6533180	best: 0.6530441 (760)	total: 4.23s	remaining: 1.05s
900:	learn: 0.4721094	test: 0.6518925	best: 0.6518925 (900)	total: 4.71s	remaining: 518ms
999:	learn: 0.4579441	test: 0.6509761	best: 0.6509761 (999)	total: 5.18s	remaining: 0us

bestTest = 0.6509